# Eksplorasi AutoModelForQuestionAnswering

**Task**: Extractive Question Answering (QA)
**Cara Kerja**: Menerima suatu paragraf sebagai Konteks (Context) dan sebuah Pertanyaan (Question). Model kemudian tidak meng-*generate* jawaban baru, melainkan mencari dan memberitahu **di mana posisi indeks awal dan akhir dari teks** di dalam konteks tersebut yang merupakan jawaban yang tepat.
**Model Populer**: BERT, RoBERTa, ALBERT.
**Dataset**: `squad` (Stanford Question Answering Dataset) - dataset standar industri untuk melatih model menjawab pertanyaan ekstraktif.

In [1]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from datasets import load_dataset
import torch

## 1. Load Dataset Publik (`squad`)
Dataset SQuAD versi 1.1 memiliki setiap baris data yang berisi: 
1. `context` (teks panjang)
2. `question` (pertanyaan terkait context)
3. `answers` (teks jawaban pasti, dan lokasi pasti pada karakter index keberapanya di dalam `context` di mana teks itu berada).

In [2]:
dataset = load_dataset("squad", split="train")

print("--- Contoh Data Index-0 ---")
print("KONTEKS:")
context = dataset[0]['context']
print(context[:400], "...\n")

print("PERTANYAAN:")
print(dataset[0]['question'], "\n")

print("JAWABAN DI DATASET (Dan index letaknya):")
print(dataset[0]['answers'])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

--- Contoh Data Index-0 ---
KONTEKS:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of p ...

PERTANYAAN:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? 

JAWABAN DI DATASET (Dan index letaknya):
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [7]:
dataset[0]['context'][515:515+len(dataset[0]['answers']['text'][0])]

'Saint Bernadette Soubirous'

## 2. Load Tokenizer & Model
Kita akan memuat model `distilbert-base-cased-distilled-squad`. DistilBERT merupakan versi kompresi dari BERT yang sangat cepat tapi akurat. Karena ekor namanya ada `distilled-squad`, berarti dia sudah pernah dilatih khusus di atas Task SQuAD.

In [8]:
model_checkpoint = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Wajib menggunakan AutoModelForQuestionAnswering
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [9]:
print(model)

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

## 3. Inferensi Manual (Menemukan Start & End Token)
Model QA tidak menghasilkan kata. Model melainkan menghasilkan 2 list probabilitas logika: 1 probabilitas Logits Start Token dan 1 probabilitas Logits End Token.

In [61]:
question = "Who is the founder of Hugging Face?"
context = "Hugging Face was founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf in New York City."

# Tokenizer pada QA dapat menerima text (sebagai quesiton) dan text_pair (sebagai context)
inputs = tokenizer(question, context, return_tensors="pt", return_offsets_mapping=True)
print(inputs.sequence_ids(0))

with torch.no_grad():
    outputs = model(**inputs)

# Kita dapatkan posisi index token dengan probabilitas paling tinggi untuk Jawaban Mulai dan Selesai
start_index = torch.argmax(outputs.start_logits)
end_index = torch.argmax(outputs.end_logits)

# Konversi id token dari posisi awal hingga akhir menjadi bahasa yang bisa dibaca manusia
predict_answer_tokens = inputs.input_ids[0, start_index : end_index + 1]
predicted_answer = tokenizer.decode(predict_answer_tokens)

print("--- Hasil Prediksi Manual ---")
print("Tugas QA Menghitung Posisi:")
print(f"Start Token berada di index: {start_index.item()}")
print(f"End Token berada di index: {end_index.item()}")
print(f"Jawaban: {predicted_answer}")

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, None]
--- Hasil Prediksi Manual ---
Tugas QA Menghitung Posisi:
Start Token berada di index: 19
End Token berada di index: 33
Jawaban: Clément Delangue, Julien Chaumond, and Thomas Wolf


## 4. Persiapan Data Pelatihan Extractive QA (PyTorch)
Data preparation untuk model QA adalah salah satu yang **paling kompleks** di antara NLP task lainnya. 

Dataset SQuAD memberikan letak jawaban dalam wujud *Karakter Ke-X* (`answer_start`). Tetapi model QA bekerja pada *level Token* (menggunakan dimensi `start_positions` & `end_positions`). Sehingga proses Tokenizer kita harus mencari letak irisan indeks token mana yang bersentuhan dengan karakter jawaban tersebut lewat bantuan properti `return_offsets_mapping=True`.

In [66]:
tes = [None, 0, 0, None, 1, 1, 1, None]
tes[::-1].index(1)

1

In [69]:
from torch.utils.data import Dataset, DataLoader

# Ambil sampel mini biar cepat
train_sample = dataset.select(range(50))

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=128,
        truncation="only_second", # Hanya potong konteks, bukan pertanyaan
        padding="max_length",
        return_offsets_mapping=True, # Wajib agar kita tau pemetaan char-to-token
    )

    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = examples["answers"][i]
        
        # Jika array jawaban kosongan
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
            
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        
        # Temukan di manakah Teks Konteks berada di dalam array Tokens
        sequence_ids = inputs.sequence_ids(i)

        context_start = sequence_ids.index(1) 
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)
        
        # Cek apakah jawaban terpotong oleh `max_length` kita
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Shift maju hingga index bersentuhan dengan start karakter aslinya
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            # Shift mundur untuk index akhir
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [70]:
tokenized_train = train_sample.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

class QADataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "start_positions": torch.tensor(item["start_positions"]), # Target Start!
            "end_positions": torch.tensor(item["end_positions"])      # Target End!
        }

train_dataloader = DataLoader(QADataset(tokenized_train), batch_size=4, shuffle=True)
print(f"Total Batch QA: {len(train_dataloader)}")
print("Selesai memetakan Karakter ke Token Index Target!")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Total Batch QA: 13
Selesai memetakan Karakter ke Token Index Target!


In [71]:
for data in train_dataloader:
    print(data['start_positions'])
    print(data['end_positions'])
    break

tensor([ 44,  53,  34, 101])
tensor([ 46,  57,  37, 107])


## 5. Proses PyTorch Training Loop
Loop pelatihannya mengoptimalkan 2 loss sekaligus yang sudah dirangkum oleh class `.from_pretrained()` milik transformer, yaitu akumulasi rata-rata *CrossEntropy* dari `start_logits` dan *CrossEntropy* dari `end_logits` yang disandingkan dengan *positions* targets-nya.

In [72]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training Extractive QA ===")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        # Eksekusi variabel ke device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_positions = batch["start_positions"].to(device)
        end_positions = batch["end_positions"].to(device)
        
        outputs = model(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            start_positions=start_positions, 
            end_positions=end_positions
        )
        # Loss gabungan = (loss(start_logits, start_pos) + loss(end_logits, end_pos)) / 2
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss Gabungan: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

==== Memulai Training Extractive QA ===
Epoch 1 | Step 0 | Loss Gabungan: 0.0494
Epoch 1 | Step 5 | Loss Gabungan: 0.0801
Epoch 1 | Step 10 | Loss Gabungan: 0.3885
>> Rata-rata Train Loss Epoch 1: 0.3823



## 6. Evaluasi Metrik: Exact Match (EM)
Salah satu metrik universal utama untuk SQuAD Task adalah Exact Match. Ini berarti persentase rasio dimana tebakan Logit Start dan End yang kita prediksi **100%** jatuh dan cocok tepat mendarat di index Target Start dan End secara persis. Jika sedikit saja bergeser 1 index panjang token tebakannya, dianggap gagal/False.

In [73]:
# Simulasi validation
val_sample = dataset.select(range(50, 70))
tokenized_val = val_sample.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
val_dataloader = DataLoader(QADataset(tokenized_val), batch_size=4)

model.eval()
exact_match = 0
total_QA = 0

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        start_true = batch["start_positions"].to(device)
        end_true = batch["end_positions"].to(device)
        
        # Prediksi tanpa labels
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Ambil probabilitas tertinggi menjadi id posisi Start & End
        start_pred = outputs.start_logits.argmax(dim=-1)
        end_pred = outputs.end_logits.argmax(dim=-1)
        
        # Syarat skor naik: MULAI dan BERAKHIR harus dua-duanya tepat secara simultan
        match = ((start_pred == start_true) & (end_pred == end_true))
        exact_match += match.sum().item()
        total_QA += input_ids.size(0)

accuracy_em = exact_match / total_QA
print(f"Evaluasi Tingkat Exact Match (EM) : {accuracy_em:.4f} atau {accuracy_em*100:.2f}%")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Evaluasi Tingkat Exact Match (EM) : 0.7500 atau 75.00%


## 7. Inferensi Hasil Akhir (Post-Training)
Setelah di-train dengan soal-soal, mari kita tes performa memori terbarunya dengan menanyakannya sebuah prompt khusus.

In [74]:
question = "Where is Hugging Face located?"
context = "Hugging Face is an AI company originally founded in New York City by some awesome people."

inputs = tokenizer(question, context, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits)

answer_tokens = inputs.input_ids[0, start_idx : end_idx + 1]
predicted_answer = tokenizer.decode(answer_tokens)

print("--- Hasil Inferensi POST-TRAIN ---")
print("Konteks   :", context)
print("Pertanyaan:", question)
print("---------------------------------")
print(f"Jawaban   : [{predicted_answer}]")

--- Hasil Inferensi POST-TRAIN ---
Konteks   : Hugging Face is an AI company originally founded in New York City by some awesome people.
Pertanyaan: Where is Hugging Face located?
---------------------------------
Jawaban   : [New York City]
